# Figure 1 — First dated VEI 5+ eruption and all 50-year follow-up eruptions

## 1. Imports and shared 00_Setup

In [1]:
from pathlib import Path
import calendar
import hashlib

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from PIL import Image

SINGLE_COLUMN_WIDTH_IN = 3.55
DOUBLE_COLUMN_WIDTH_IN = 7.25
MAXIMUM_FIGURE_HEIGHT_IN = 7.8
FIGURE_WIDTH_IN = 3.5
FIGURE_HEIGHT_IN = 3.5
BASE_TEXT_SIZE_PT = 8.0
AXIS_LABEL_SIZE_PT = 8.0
# The compact single-column panel uses the 00_Setup minimum readable text size.
LEGEND_SIZE_PT = 6.0
LEGEND_TITLE_SIZE_PT = 8.0
MINIMUM_TEXT_SIZE_PT = 6.0
MINIMUM_LINE_WIDTH_PT = 0.28
EXPORT_DPI = 300
PUBLICATION_FONT = "Arial"

installed_font_names = {entry.name for entry in font_manager.fontManager.ttflist}
if PUBLICATION_FONT not in installed_font_names:
    raise RuntimeError("Arial is required by 00_Setup.")
mpl.rcParams.update({
    "font.family":"sans-serif", "font.sans-serif":[PUBLICATION_FONT],
    "font.size":BASE_TEXT_SIZE_PT, "axes.labelsize":AXIS_LABEL_SIZE_PT,
    "xtick.labelsize":BASE_TEXT_SIZE_PT, "ytick.labelsize":BASE_TEXT_SIZE_PT,
    "legend.fontsize":LEGEND_SIZE_PT, "legend.title_fontsize":LEGEND_TITLE_SIZE_PT, "figure.facecolor":"white",
    "axes.facecolor":"white", "axes.spines.top":True, "axes.spines.right":True,
    "axes.linewidth":0.5, "lines.linewidth":0.75,
    "xtick.direction":"in", "ytick.direction":"in",
    "xtick.major.width":0.5, "ytick.major.width":0.5,
    "savefig.dpi":EXPORT_DPI, "savefig.facecolor":"white",
    "savefig.transparent":False, "pdf.fonttype":42, "ps.fonttype":42,
    "svg.fonttype":"none",
})


## 2. Paths and public supplemental workbooks

In [2]:
PROJECT_ROOT_OVERRIDE = None
PROJECT_ROOT = PROJECT_ROOT_OVERRIDE
if PROJECT_ROOT is None:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "SA_MI_Draft_FinalFigures_Work" / "DataSheets").is_dir():
            PROJECT_ROOT = candidate
            break
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the project or set PROJECT_ROOT_OVERRIDE.")
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
WORK_ROOT = PROJECT_ROOT / "SA_MI_Draft_FinalFigures_Work"
DATA_DIRECTORY = WORK_ROOT / "DataSheets" / "Updated_Oct20"
FIGURE_DIRECTORY = WORK_ROOT / "Nov6_AllFigures" / "Figure_01_VEI"
OUTPUT_DIRECTORY = FIGURE_DIRECTORY / "Figure_01_VEI_V3_Final"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

S1_PATH = DATA_DIRECTORY / "Dechert_S1_AllData.xlsx"
S2_PATH = DATA_DIRECTORY / "Dechert_S2_Standards.xlsx"
S3_PATH = DATA_DIRECTORY / "Dechert_S3_Compilations.xlsx"
for workbook_path in [S1_PATH, S2_PATH, S3_PATH]:
    if not workbook_path.is_file():
        raise FileNotFoundError(f"Missing supplemental workbook: {workbook_path}")
SOURCE_HASHES_BEFORE = {
    "S1": hashlib.sha256(S1_PATH.read_bytes()).hexdigest(),
    "S2": hashlib.sha256(S2_PATH.read_bytes()).hexdigest(),
    "S3": hashlib.sha256(S3_PATH.read_bytes()).hexdigest(),
}


## 3. Select the first dated VEI 5+ eruption and build each sequence

In [ ]:
raw = pd.read_excel(S3_PATH, sheet_name="GVP 2024")
assert raw["Eruption Number"].notna().all()
assert not raw["Eruption Number"].duplicated().any()

data = raw.copy().replace("NULL", np.nan)
numeric_columns = [
    "Volcano Number", "Eruption Number", "VEI", "Start Year",
    "Start Year Uncertainty", "Start Month", "Start Day",
    "Start Day Uncertainty", "End Year",
]
for column in numeric_columns:
    data[column] = pd.to_numeric(data[column], errors="coerce")

start_time = []
start_lower = []
start_upper = []
date_precision = []
for _, row in data.iterrows():
    year = row["Start Year"]
    if pd.isna(year):
        start_time.append(np.nan)
        start_lower.append(np.nan)
        start_upper.append(np.nan)
        date_precision.append("missing")
        continue
    year = int(year)
    astronomical_year = year + 1 if year < 0 else year
    month = row["Start Month"]
    day = row["Start Day"]
    days_in_year = 366 if calendar.isleap(astronomical_year) else 365
    if pd.isna(month) or month == 0:
        lower = float(astronomical_year)
        upper = float(astronomical_year + 1)
        precision = "year"
    elif not (1 <= month <= 12):
        start_time.append(np.nan)
        start_lower.append(np.nan)
        start_upper.append(np.nan)
        date_precision.append("invalid month")
        continue
    else:
        month = int(month)
        month_offset = sum(calendar.monthrange(astronomical_year, number)[1] for number in range(1, month))
        days_in_month = calendar.monthrange(astronomical_year, month)[1]
        if pd.isna(day) or day == 0:
            lower = astronomical_year + month_offset / days_in_year
            upper = astronomical_year + (month_offset + days_in_month) / days_in_year
            precision = "month"
        elif 1 <= day <= days_in_month:
            lower = astronomical_year + (month_offset + int(day) - 1) / days_in_year
            upper = lower + 1 / days_in_year
            precision = "day"
        else:
            start_time.append(np.nan)
            start_lower.append(np.nan)
            start_upper.append(np.nan)
            date_precision.append("invalid day")
            continue
    year_uncertainty = abs(row["Start Year Uncertainty"]) if pd.notna(row["Start Year Uncertainty"]) else 0
    day_uncertainty = abs(row["Start Day Uncertainty"]) / days_in_year if pd.notna(row["Start Day Uncertainty"]) else 0
    start_time.append((lower + upper) / 2)
    start_lower.append(lower - year_uncertainty - day_uncertainty)
    start_upper.append(upper + year_uncertainty + day_uncertainty)
    date_precision.append(precision)

data["start_time"] = start_time
data["start_lower"] = start_lower
data["start_upper"] = start_upper
data["date_precision"] = date_precision
data["date_modified"] = data[["Start Year Modifier", "Start Day Modifier"]].notna().any(axis=1)
data["vei_modified"] = data["VEI Modifier"].notna()

ordered = data.sort_values(["Volcano Number", "start_time", "Eruption Number"], na_position="last").copy()
ordered["sequence_has_undated"] = (
    ordered.groupby("Volcano Number")["start_time"].transform("count")
    < ordered.groupby("Volcano Number")["start_time"].transform("size")
)
ordered["start_tied"] = ordered["start_time"].notna() & ordered.duplicated(["Volcano Number", "start_time"], keep=False)
next_fields = [
    "Eruption Number", "VEI", "VEI Modifier", "Start Year",
    "Start Year Uncertainty", "start_time", "start_lower",
    "start_upper", "date_precision", "date_modified",
    "vei_modified", "start_tied",
]
pairs = ordered.copy()
for column in next_fields:
    pairs["next_" + column] = ordered.groupby("Volcano Number")[column].shift(-1)
pairs["has_successor"] = pairs["next_Eruption Number"].notna()
pairs["delta_t_years"] = pairs["next_start_time"] - pairs["start_time"]
pairs["delta_t_lower"] = pairs["next_start_lower"] - pairs["start_upper"]
pairs["delta_t_upper"] = pairs["next_start_upper"] - pairs["start_lower"]
pairs["delta_vei"] = pairs["next_VEI"] - pairs["VEI"]
pairs["order_ambiguous"] = pairs["start_tied"] | pairs["next_start_tied"].eq(True) | pairs["delta_t_lower"].le(0)
pairs["pair_modified"] = (
    pairs["date_modified"] | pairs["vei_modified"]
    | pairs["next_date_modified"].eq(True) | pairs["next_vei_modified"].eq(True)
)

start_mask = pairs["VEI"].ge(5)
starts = pairs.loc[start_mask].copy()
starts["plot_eligible"] = (
    starts["has_successor"] & starts["next_VEI"].notna()
    & starts["delta_t_years"].gt(0) & ~starts["order_ambiguous"]
    & ~starts["sequence_has_undated"]
)
plot_data = starts.loc[starts["plot_eligible"]].copy()


# Select one first dated VEI 5+ anchor per volcano.
EXCLUDE_VOLCANOES = ["Katla", "Hekla", "Askja", "Agua de Pau"] #Not in arcs 
anchor_data = plot_data.loc[
    plot_data["delta_t_years"].le(50.0)
    & ~plot_data["Volcano Name"].isin(EXCLUDE_VOLCANOES)
].copy()
anchor_data = (
    anchor_data.sort_values(["Volcano Number", "start_time", "Eruption Number"], kind="stable")
    .groupby("Volcano Number", sort=False).head(1).copy()
)

sequence_rows = []
for _, anchor in anchor_data.iterrows():
    volcano_data = ordered.loc[ordered["Volcano Number"].eq(anchor["Volcano Number"])].copy()
    sequence = volcano_data.loc[
        volcano_data["start_time"].notna()
        & volcano_data["start_time"].ge(anchor["start_time"])
        & volcano_data["start_time"].le(anchor["start_time"] + 50.0)
        & volcano_data["VEI"].notna()
    ].copy()
    sequence["is_anchor"] = sequence["Eruption Number"].eq(anchor["Eruption Number"])
    sequence = sequence.sort_values(["start_time", "is_anchor", "Eruption Number"], ascending=[True, False, True], kind="stable")
    sequence["elapsed_years"] = sequence["start_time"] - anchor["start_time"]
    sequence["anchor_eruption_number"] = anchor["Eruption Number"]
    sequence_rows.append(sequence)

sequence_data = pd.concat(sequence_rows, ignore_index=True)
sequence_data["elapsed_years"] = sequence_data["elapsed_years"].clip(lower=0.0, upper=50.0)
sequence_data["assumed"] = False

gp_assumed = pd.DataFrame({
    "Volcano Number":[321020,321020], "Volcano Name":["Glacier Peak","Glacier Peak"],
    "Eruption Number":[-1,-2], "VEI":[5.0,5.0], "elapsed_years":[0.0,10.0],
    "is_anchor":[True,False], "anchor_eruption_number":[-1,-1], "assumed":[True,True],
}) #Adding assumed Glacier Peak eruptions at 0 and 10 years to illustrate the potential for a second eruption within 10 years of the first.
sequence_data = pd.concat([sequence_data, gp_assumed], ignore_index=True)


## 4. Plot all follow-up eruptions within 50 years

In [4]:
STYLES = {
    "Glacier Peak":{"color":"#ff1493","marker":"o"}, "St. Helens":{"color":"#5145CD","marker":"*"},
    "Pinatubo":{"color":"#C1121F","marker":">"}, "Vesuvius":{"color":"#8C510A","marker":"s"},
    "Krakatau":{"color":"#E66101","marker":"D"}, "Hudson, Cerro":{"color":"#008577","marker":"^"},
    "Tambora":{"color":"#667A16","marker":"P"},
}
DEFAULT_STYLE = {"color":"#A0A0A0","marker":"."}
HIGHLIGHTS = ["Vesuvius", "Krakatau", "Hudson, Cerro", "Tambora", "St. Helens", "Pinatubo", "Glacier Peak"]
DRAW_RANK = {name:index + 1 for index,name in enumerate(HIGHLIGHTS)}

figure, axis = plt.subplots(figsize=(FIGURE_WIDTH_IN, FIGURE_HEIGHT_IN))
figure.subplots_adjust(left=0.16, right=0.98, bottom=0.15, top=0.97)
axis.set(xlim=(0,50), ylim=(0,7.35), xlabel="Elapsed Time (years)", ylabel="Volcano Explosivity Index (VEI)")
axis.set_xticks(np.arange(0,51,10)); axis.set_yticks(range(8)); axis.tick_params(length=2.5,pad=2); axis.set_axisbelow(True)
axis.grid(True,which="major",axis="both",color="0.90",linewidth=0.35)
for spine in axis.spines.values(): spine.set_visible(True); spine.set_color("black"); spine.set_linewidth(0.5)

for anchor_number, group in sequence_data.groupby("anchor_eruption_number", sort=False):
    group = group.sort_values(["elapsed_years","is_anchor","Eruption Number"], ascending=[True,False,True], kind="stable")
    volcano_name = group["Volcano Name"].iloc[0]; style = STYLES.get(volcano_name,DEFAULT_STYLE); highlighted = volcano_name in HIGHLIGHTS
    zorder = 2 + 4 * DRAW_RANK.get(volcano_name,0)
    line_width = 1.15 if volcano_name == "Glacier Peak" else 0.85 if highlighted else 0.5
    marker_size = 5.5 if volcano_name == "Glacier Peak" else 4.5 if highlighted else 2.6
    axis.plot(group["elapsed_years"],group["VEI"],color=style["color"],linewidth=line_width,linestyle="--" if group["assumed"].all() else "-",alpha=1.0 if highlighted else 0.85,zorder=zorder)
    first_event = group.iloc[0]; following_events = group.iloc[1:]
    axis.plot(first_event["elapsed_years"],first_event["VEI"],marker=style["marker"],markersize=marker_size,markerfacecolor="white",markeredgecolor=style["color"],markeredgewidth=0.5,linestyle="none",zorder=zorder+1,clip_on=False)
    axis.plot(following_events["elapsed_years"],following_events["VEI"],marker=style["marker"],markersize=marker_size,markerfacecolor=style["color"],markeredgecolor="black" if highlighted else style["color"],markeredgewidth=MINIMUM_LINE_WIDTH_PT,linestyle="none",zorder=zorder+2,clip_on=False)

legend_handles = [Line2D([],[],color=STYLES[name]["color"],marker=STYLES[name]["marker"],markersize=5 if name == "Glacier Peak" else 4,linestyle="--" if name == "Glacier Peak" else "-",linewidth=1.1 if name == "Glacier Peak" else 0.75,label="Hudson" if name == "Hudson, Cerro" else "Mount St. Helens" if name == "St. Helens" else name) for name in HIGHLIGHTS]
legend_handles.extend([Line2D([],[],color=DEFAULT_STYLE["color"],linewidth=0.5,label="Other volcanoes"),Line2D([],[],marker="o",markersize=3.5,markerfacecolor="white",markeredgecolor="black",markeredgewidth=0.5,linestyle="none",label="First eruption"),Line2D([],[],marker="o",markersize=3.5,markerfacecolor="black",markeredgecolor="black",linestyle="none",label="Following eruptions")])
axis.legend(handles=legend_handles,loc="upper right",fontsize=LEGEND_SIZE_PT,ncol=2,frameon=True,fancybox=False,framealpha=1.0,edgecolor="black",borderpad=0.45,labelspacing=0.3,columnspacing=0.8,handlelength=1.2)

FIGURE_STEM = "Figure_01_VEI_V3"
EXPORT_PATHS = {}
for output_format in ["pdf","png","svg"]:
    output_path = OUTPUT_DIRECTORY / f"{FIGURE_STEM}.{output_format}"; save_options = {"format":output_format,"facecolor":"white","transparent":False}
    if output_format == "png": save_options["dpi"] = EXPORT_DPI
    figure.savefig(output_path,**save_options)
    if not output_path.is_file() or output_path.stat().st_size == 0: raise IOError(f"Expected export was not created: {output_path}")
    EXPORT_PATHS[output_format] = output_path
plt.close(figure)
